# LeetCode #78 — Subsets: ba cách sinh tập lũy thừa (power set)

**Mức độ:** Medium · **Chủ đề:** Array, Backtracking, Bit Manipulation

Skeleton trên LeetCode:

```python
class Solution:
    def subsets(self, nums: List[int]) -> List[List[int]]:
```

Mục tiêu của notebook:

1. Hiểu vì sao có đúng `2^n` tập con.
2. Cách 1 — **lặp (cascading):** mỗi phần tử nhân đôi danh sách hiện có.
3. Cách 2 — **backtracking:** cây "lấy / không lấy", hoặc cây "chọn phần tử tiếp theo".
4. Cách 3 — **bitmask:** mỗi số `0 .. 2^n − 1` ứng với một tập con.

# 1. Bài toán là gì?

Cho mảng `nums` gồm các số **phân biệt**, trả về **mọi tập con** (power set). Không được có tập con trùng; thứ tự tùy ý.

**Constraints (theo đề):**

- `1 <= nums.length <= 10`
- `-10 <= nums[i] <= 10`
- Mọi phần tử của `nums` **phân biệt**.

---

## Ví dụ

```text
nums = [1, 2, 3]  ->  [[], [1], [2], [1,2], [3], [1,3], [2,3], [1,2,3]]
nums = [0]        ->  [[], [0]]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quay lui (backtracking)
2. Biểu diễn nhị phân và các phép toán bit
3. Hoán vị, tổ hợp, tập con và thứ tự từ điển

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [1]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [2]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [3]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [4]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]: ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [5]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép: [[], [], [], []]


## 2.2 Biểu diễn nhị phân và các phép toán bit

### Số nguyên dưới dạng bit

Máy tính lưu số nguyên ở hệ **nhị phân**: mỗi chữ số (bit) là 0 hoặc 1, bit thứ `k` (đếm từ 0, bên phải) có giá trị `2^k`.

```text
13 = 8 + 4 + 1 = 1·2³ + 1·2² + 0·2¹ + 1·2⁰ = 0b1101
bit:              3     2      1      0
```

In [6]:
x_bit = 13
print("bin(13) =", bin(x_bit), "| format 8 bit:", format(x_bit, "08b"), "| int('1101', 2) =", int("1101", 2))
print("bit thứ k của 13:", [(x_bit >> k) & 1 for k in range(4)], "(từ bit 0)")

bin(13) = 0b1101 | format 8 bit: 00001101 | int('1101', 2) = 13
bit thứ k của 13: [1, 0, 1, 1] (từ bit 0)


### Sáu phép toán bit

| Phép | Ký hiệu | Quy tắc từng bit | Ví dụ `12 = 1100`, `10 = 1010` |
|---|---|---|---|
| AND | `a & b` | 1 nếu **cả hai** là 1 | `1000` = 8 |
| OR | `a \| b` | 1 nếu **ít nhất một** là 1 | `1110` = 14 |
| XOR | `a ^ b` | 1 nếu **khác nhau** | `0110` = 6 |
| NOT | `~a` | đảo mọi bit | `~12 = -13` (Python) |
| dịch trái | `a << k` | thêm `k` số 0 bên phải = nhân `2^k` | `12 << 1 = 24` |
| dịch phải | `a >> k` | bỏ `k` bit phải = chia nguyên `2^k` | `12 >> 2 = 3` |

In [7]:
a_b, b_b = 12, 10
for ten, v in [("a & b", a_b & b_b), ("a | b", a_b | b_b), ("a ^ b", a_b ^ b_b), ("a << 1", a_b << 1), ("a >> 2", a_b >> 2), ("~a", ~a_b)]:
    print(f"{ten:<7} = {v:>4}   bin: {bin(v)}")

a & b   =    8   bin: 0b1000
a | b   =   14   bin: 0b1110
a ^ b   =    6   bin: 0b110
a << 1  =   24   bin: 0b11000
a >> 2  =    3   bin: 0b11
~a      =  -13   bin: -0b1101


### Các mẹo hay dùng

| Mục đích | Biểu thức |
|---|---|
| Kiểm tra bit `k` | `(x >> k) & 1` |
| Bật / tắt / lật bit `k` | `x \| (1 << k)`, `x & ~(1 << k)`, `x ^ (1 << k)` |
| Số chẵn? | `x & 1 == 0` |
| Bỏ bit 1 thấp nhất | `x & (x - 1)` |
| Lấy bit 1 thấp nhất | `x & -x` |
| Là lũy thừa của 2? | `x > 0 and x & (x - 1) == 0` |

**Tính chất XOR** (cực kỳ hay dùng): `x ^ x = 0`, `x ^ 0 = x`, giao hoán và kết hợp → XOR cả dãy thì các số xuất hiện **chẵn** lần triệt tiêu nhau.

In [8]:
x_m = 0b10110
print("bỏ bit 1 thấp nhất:", format(x_m, "05b"), "→", format(x_m & (x_m - 1), "05b"))
print("lấy bit 1 thấp nhất:", format(x_m & -x_m, "05b"))
print("lũy thừa của 2:", [v for v in range(1, 40) if v & (v - 1) == 0])
print("đếm bit 1 của 255:", bin(255).count("1"))
xor_all = 0
for v in [4, 1, 2, 1, 2]:
    xor_all ^= v
print("XOR [4,1,2,1,2] =", xor_all, "(số xuất hiện lẻ lần)")

bỏ bit 1 thấp nhất: 10110 → 10100
lấy bit 1 thấp nhất: 00010
lũy thừa của 2: [1, 2, 4, 8, 16, 32]
đếm bit 1 của 255: 8
XOR [4,1,2,1,2] = 4 (số xuất hiện lẻ lần)


### Dùng bit làm tập hợp (bitmask)

Với `n` phần tử, một số `mask` từ `0` tới `2^n − 1` biểu diễn một **tập con**: bit `i` bằng 1 nghĩa là lấy phần tử `i`. Duyệt mọi `mask` = duyệt mọi tập con.

In [9]:
items = ["a", "b", "c"]
for mask in range(1 << len(items)):
    tap = [items[i] for i in range(len(items)) if mask >> i & 1]
    print(format(mask, "03b"), tap)

000 []
001 ['a']
010 ['b']
011 ['a', 'b']
100 ['c']
101 ['a', 'c']
110 ['b', 'c']
111 ['a', 'b', 'c']


### Số âm và giới hạn 32 bit

Ngôn ngữ như C/Java lưu `int` bằng 32 bit, số âm dạng **bù hai** (two's complement): `-x = ~x + 1`. Python có số nguyên **không giới hạn**, nên muốn mô phỏng 32 bit phải **che** bằng `& 0xFFFFFFFF` rồi đổi lại thành số có dấu.

In [10]:
def thanh_32bit(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >> 31 else x       # bit 31 là bit dấu

print("-1 ở 32 bit:", format(-1 & 0xFFFFFFFF, "032b"))
print("2^31 tràn thành:", thanh_32bit(2 ** 31), "| (2^31 - 1) + 1 =", thanh_32bit(2 ** 31 - 1 + 1))

-1 ở 32 bit: 11111111111111111111111111111111
2^31 tràn thành: -2147483648 | (2^31 - 1) + 1 = -2147483648


Lỗi hay gặp: nhầm `^` là lũy thừa (lũy thừa là `**`); quên ngoặc vì `&`, `|`, `^` có độ ưu tiên **thấp hơn** `==` — viết `(x & 1) == 0`; `~x` trong Python là `-x - 1`, không phải "đảo 32 bit".

## 2.3 Hoán vị, tổ hợp, tập con và thứ tự từ điển

### Đếm

| Đối tượng | Ý nghĩa | Số lượng |
|---|---|---|
| Hoán vị của `n` phần tử | sắp **thứ tự** tất cả | `n!` |
| Chỉnh hợp chập `k` | chọn `k` phần tử **có thứ tự** | `n! / (n−k)!` |
| Tổ hợp chập `k` | chọn `k` phần tử **không** thứ tự | `C(n, k) = n! / (k! (n−k)!)` |
| Tập con | mỗi phần tử lấy/không | `2^n` |
| Hoán vị có lặp | `n` phần tử, nhóm trùng cỡ `c1, c2, …` | `n! / (c1! c2! …)` |

In [11]:
from math import factorial, comb, perm
from itertools import permutations, combinations, product

print("4! =", factorial(4), "| P(4,2) =", perm(4, 2), "| C(4,2) =", comb(4, 2), "| 2^4 =", 2 ** 4)
print("hoán vị [1,2,3]:", list(permutations([1, 2, 3])))
print("tổ hợp chập 2 của [1,2,3,4]:", list(combinations([1, 2, 3, 4], 2)))
print("hoán vị khác nhau của 'aab':", sorted(set(permutations("aab"))), "=", factorial(3) // factorial(2))
print("tích Descartes {0,1}^2:", list(product([0, 1], repeat=2)))

4! = 24 | P(4,2) = 12 | C(4,2) = 6 | 2^4 = 16
hoán vị [1,2,3]: [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
tổ hợp chập 2 của [1,2,3,4]: [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
hoán vị khác nhau của 'aab': [('a', 'a', 'b'), ('a', 'b', 'a'), ('b', 'a', 'a')] = 3
tích Descartes {0,1}^2: [(0, 0), (0, 1), (1, 0), (1, 1)]


### Thứ tự từ điển (lexicographic order)

So hai dãy như so từ trong từ điển: so phần tử **đầu tiên khác nhau**; nếu một dãy là tiền tố của dãy kia, dãy ngắn hơn đứng trước. Python so sánh list/tuple/str đúng theo kiểu này.

Liệt kê hoán vị của `[1, 2, 3]` theo thứ tự từ điển: `123, 132, 213, 231, 312, 321`.

**Quan sát về hoán vị kế tiếp:** đi từ phải sang, phần **hậu tố giảm dần** dài nhất là hậu tố "đã lớn nhất có thể". Muốn hoán vị lớn hơn kế tiếp, phải tăng phần tử **ngay trước** hậu tố đó lên một chút.

In [12]:
print([1, 2, 3] < [1, 3], "|", "abc" < "abd", "|", [1, 2] < [1, 2, 0], "|", "10" < "9")
for p in permutations([1, 2, 3]):
    i = len(p) - 1
    while i > 0 and p[i - 1] >= p[i]:
        i -= 1
    print(p, "| hậu tố giảm dần dài nhất:", p[i:])

True | True | True | True
(1, 2, 3) | hậu tố giảm dần dài nhất: (3,)
(1, 3, 2) | hậu tố giảm dần dài nhất: (3, 2)
(2, 1, 3) | hậu tố giảm dần dài nhất: (3,)
(2, 3, 1) | hậu tố giảm dần dài nhất: (3, 1)
(3, 1, 2) | hậu tố giảm dần dài nhất: (2,)
(3, 2, 1) | hậu tố giảm dần dài nhất: (3, 2, 1)


### Liên hệ với quay lui

`itertools` sinh sẵn hoán vị/tổ hợp, nhưng phỏng vấn thường yêu cầu **tự sinh** bằng quay lui (để kèm cắt tỉa, bỏ trùng, điều kiện riêng). Kích thước kết quả (`n!`, `C(n,k)`, `2^n`) cho biết độ phức tạp tối thiểu — không thuật toán nào liệt kê nhanh hơn số lời giải.

## 2.4 Áp dụng vào bài này

Mọi **tập con** (`2^n`): quay lui (mỗi nút của cây quyết định là một tập con), hoặc **bitmask** — mỗi số `0..2^n−1` là một tập con, bit `i` cho biết có lấy phần tử `i` không.

# 3. Vì sao `2^n`?

Với mỗi phần tử, có đúng hai lựa chọn: **lấy** hoặc **không lấy**. `n` phần tử độc lập → `2 · 2 · … · 2 = 2^n` tập con (kể cả tập rỗng và chính `nums`).

Với `n = 10`: `1024` tập con — rất nhỏ, nên mọi cách liệt kê đều đủ nhanh. Điều quan trọng là viết **đúng** và **gọn**.

# 4. Cách 1: Lặp — mỗi phần tử nhân đôi danh sách

Bắt đầu với `[[]]`. Với mỗi số `x`, thêm `x` vào **mọi** tập con đang có, tạo ra các tập con mới.

```text
[]                                  ban đầu
[] [1]                              thêm 1
[] [1] [2] [1,2]                    thêm 2
[] [1] [2] [1,2] [3] [1,3] [2,3] [1,2,3]   thêm 3
```

In [13]:
from typing import List
import itertools, random

def subsets_iter(nums: List[int]) -> List[List[int]]:
    result = [[]]
    for x in nums:
        result += [s + [x] for s in result]
    return result

print(subsets_iter([1, 2, 3]))

[[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]


`result += [... for s in result]`: vế phải được tính **xong** (dựa trên `result` cũ) rồi mới nối vào — nên không bị lặp vô hạn.

# 5. Cách 2: Backtracking

## 5.1 Cây "lấy / không lấy"

Ở tầng `i`, quyết định lấy hay không lấy `nums[i]`. Lá ở tầng `n` là một tập con.

In [14]:
def subsets_take_skip(nums: List[int]) -> List[List[int]]:
    result, path = [], []
    def dfs(i):
        if i == len(nums):
            result.append(path[:])
            return
        dfs(i + 1)                 # không lấy nums[i]
        path.append(nums[i])       # lấy nums[i]
        dfs(i + 1)
        path.pop()
    dfs(0)
    return result

print(subsets_take_skip([1, 2, 3]))

[[], [3], [2], [2, 3], [1], [1, 3], [1, 2], [1, 2, 3]]


## 5.2 Cây "chọn phần tử tiếp theo" (giống #77, nhưng lưu ở **mọi** nút)

Mỗi nút của cây tổ hợp (#77) đều là một tập con — không chỉ các lá.

In [15]:
def subsets_backtrack(nums: List[int]) -> List[List[int]]:
    result, path = [], []
    def backtrack(start):
        result.append(path[:])         # mọi nút đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()
    backtrack(0)
    return result

print(subsets_backtrack([1, 2, 3]))

[[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]


In [16]:
def trace_subsets(nums: List[int]):
    result, path = [], []
    def backtrack(start, depth):
        print("  " * depth + f"lưu {path}")
        result.append(path[:])
        for i in range(start, len(nums)):
            path.append(nums[i])
            backtrack(i + 1, depth + 1)
            path.pop()
    backtrack(0, 0)
    return result

_ = trace_subsets([1, 2, 3])

lưu []
  lưu [1]
    lưu [1, 2]
      lưu [1, 2, 3]
    lưu [1, 3]
  lưu [2]
    lưu [2, 3]
  lưu [3]


Thụt lề = độ sâu = kích thước tập con. Mỗi dòng "lưu" là một nút; có đúng `2^3 = 8` nút.

# 6. Cách 3: Bitmask

Đánh số các phần tử `0 .. n−1`. Một số nguyên `mask` trong `[0, 2^n)` có `n` bit; bit thứ `i` bằng 1 ⇔ lấy `nums[i]`.

| mask | nhị phân (bit 2,1,0) | tập con của `[1,2,3]` |
|---|---|---|
| 0 | `000` | `[]` |
| 1 | `001` | `[1]` |
| 2 | `010` | `[2]` |
| 3 | `011` | `[1,2]` |
| 5 | `101` | `[1,3]` |
| 7 | `111` | `[1,2,3]` |

Kiểm tra bit `i`: `mask >> i & 1` (hoặc `mask & (1 << i)`).

In [17]:
def subsets_bitmask(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    return [[nums[i] for i in range(n) if mask >> i & 1] for mask in range(1 << n)]

for mask, s in enumerate(subsets_bitmask([1, 2, 3])):
    print(f"mask = {mask} = {mask:03b} -> {s}")

mask = 0 = 000 -> []
mask = 1 = 001 -> [1]
mask = 2 = 010 -> [2]
mask = 3 = 011 -> [1, 2]
mask = 4 = 100 -> [3]
mask = 5 = 101 -> [1, 3]
mask = 6 = 110 -> [2, 3]
mask = 7 = 111 -> [1, 2, 3]


`1 << n` bằng `2^n`. Cách bitmask rất gọn và **không đệ quy**; hữu ích khi cần duyệt mọi tập con trong các bài DP trên tập (bitmask DP).

# 7. Phiên bản đúng format LeetCode

Dùng cách backtracking (dễ mở rộng sang #90 Subsets II).

In [18]:
class Solution:
    def subsets(self, nums: List[int]) -> List[List[int]]:
        result, path = [], []

        def backtrack(start: int) -> None:
            result.append(path[:])
            for i in range(start, len(nums)):
                path.append(nums[i])
                backtrack(i + 1)
                path.pop()

        backtrack(0)
        return result

In [19]:
def norm(res):
    return sorted(sorted(s) for s in res)

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", norm(sol.subsets([1, 2, 3])), [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]])
check("Ví dụ 2", norm(sol.subsets([0])), [[], [0]])
check("Số âm", norm(sol.subsets([-1, 5])), [[], [-1], [-1, 5], [5]])

random.seed(78)
for _ in range(300):
    nums = random.sample(range(-10, 11), random.randint(1, 8))
    expected = norm(list(c) for r in range(len(nums) + 1) for c in itertools.combinations(nums, r))
    for f in (sol.subsets, subsets_iter, subsets_take_skip, subsets_bitmask):
        res = f(nums)
        assert len(res) == 2 ** len(nums)
        assert norm(res) == expected
print("300 mảng ngẫu nhiên: cả 4 cách cho đúng 2^n tập con, không trùng")

Ví dụ 1          PASS  [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
Ví dụ 2          PASS  [[], [0]]
Số âm            PASS  [[], [-1], [-1, 5], [5]]


300 mảng ngẫu nhiên: cả 4 cách cho đúng 2^n tập con, không trùng


# 8. Vì sao đúng? Độ phức tạp?

- **Lặp:** sau khi xử lý `x1..xj`, `result` chứa đúng mọi tập con của `{x1..xj}` (quy nạp: tập con của `{x1..x(j+1)}` hoặc không chứa `x(j+1)` — đã có — hoặc chứa nó — vừa thêm).
- **Backtracking (4.2):** song ánh giữa các nút của cây và các dãy chỉ số tăng dần, tức các tập con.
- **Bitmask:** song ánh giữa số `n` bit và tập con.

```text
Time:  O(n · 2^n)  — 2^n tập con, mỗi tập sao chép tối đa n phần tử
Space: O(n) ngoài output (đệ quy / path)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Chỉ lưu ở lá (copy nguyên code #77)

In [20]:
def wrong_only_full(nums: List[int]) -> List[List[int]]:
    result, path = [], []
    def backtrack(start):
        if start == len(nums):                  # SAI: chỉ lưu khi đi hết mảng
            result.append(path[:])
            return
        for i in range(start, len(nums)):
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()
    backtrack(0)
    return result

print("Sai :", wrong_only_full([1, 2, 3]))

Sai : [[1, 2, 3], [1, 3], [2, 3], [3]]


Chỉ thu được các tập con **chứa phần tử cuối** — thiếu `[]`, `[1]`, `[2]`, `[1,2]`.

## Lỗi 2: Sửa list trong lúc lặp qua chính nó

In [21]:
def wrong_mutate_while_iter(nums: List[int]) -> List[List[int]]:
    result = [[]]
    for x in nums:
        for s in result:                        # SAI: result dài ra trong lúc lặp
            result.append(s + [x])
            if len(result) > 50:
                return result                   # chặn để không lặp vô hạn
    return result

print("Sai : độ dài =", len(wrong_mutate_while_iter([1, 2, 3])), "| Đúng: 8")

Sai : độ dài = 51 | Đúng: 8


## Lỗi 3: `s.append(x)` thay vì `s + [x]`

`append` sửa **tại chỗ** các tập con cũ → làm hỏng chúng.

In [22]:
def wrong_append(nums: List[int]) -> List[List[int]]:
    result = [[]]
    for x in nums:
        new = []
        for s in result:
            s.append(x)                          # SAI: sửa luôn s cũ
            new.append(s)
        result += new
    return result

print("Sai :", wrong_append([1, 2]))

Sai : [[1, 2, 2], [1, 2, 2], [1, 2, 2], [1, 2, 2]]


## Lỗi 4: So sánh `mask & (1 << i)` với 1

`mask & (1 << i)` trả về **`1 << i`** (chẳng hạn 2, 4, 8…) khi bit bật, **không phải** 1. So sánh `== 1` chỉ đúng với `i = 0`.

In [23]:
def wrong_bit_check(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    return [[nums[i] for i in range(n) if mask & (1 << i) == 1] for mask in range(1 << n)]   # SAI

print("Sai :", wrong_bit_check([1, 2, 3]))
print("Đúng:", subsets_bitmask([1, 2, 3]))

Sai : [[], [1], [], [1], [], [1], [], [1]]
Đúng: [[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]


Viết `mask >> i & 1` (luôn ra 0 hoặc 1), hoặc `if mask & (1 << i):` (dùng tính "truthy"). Trong Python, `&` ưu tiên **cao hơn** `==`, nên `mask & (1 << i) == 1` được hiểu là `(mask & (1 << i)) == 1` — đúng thứ ta viết, nhưng logic vẫn sai. (Trong C/Java thì ngược lại: `==` ưu tiên cao hơn `&`, nên bắt buộc phải có ngoặc.)

# 10. Mẹo nhận diện pattern

"Liệt kê mọi tập con" → một trong ba cách trên. Nếu chỉ cần **đếm** hoặc **tối ưu** trên tập con thì thường là DP (vd. #416 Partition Equal Subset Sum), không liệt kê.

| Bài | Khác biệt |
|---|---|
| #78 Subsets | phần tử phân biệt |
| #90 Subsets II | có trùng → sắp xếp + bỏ `nums[i] == nums[i-1]` cùng tầng |
| #77 Combinations | chỉ lấy tập con kích thước `k` |
| #1863 Sum of All Subset XOR Totals | duyệt mọi tập con, tính XOR |
| #2044 Count Number of Maximum Bitwise-OR Subsets | bitmask |

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Liệt kê tập con ứng với các mask `4, 6` của `nums = [1, 2, 3]`. Theo cách lặp (mục 4), tập con `[1, 3]` xuất hiện ở vị trí thứ mấy (đếm từ 0)?

## Bài 2 — Điền chỗ trống (bitmask)

In [24]:
def exercise_2_subsets_bitmask(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    result = []
    for mask in range(1 << n):
        # TODO: xây tập con ứng với mask
        pass
    return result


# assert sorted(map(sorted, exercise_2_subsets_bitmask([1, 2, 3]))) == [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
# assert len(exercise_2_subsets_bitmask(list(range(10)))) == 1024

## Bài 3 — Biến thể: tổng XOR của mọi tập con (LeetCode #1863)

Với mỗi tập con, tính XOR các phần tử (tập rỗng = 0). Trả tổng các giá trị đó.

```python
subset_xor_sum([1, 3])        -> 6       # 0 + 1 + 3 + (1^3=2)
subset_xor_sum([5, 1, 6])     -> 28
```

In [25]:
def exercise_3_subset_xor_sum(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_3_subset_xor_sum([1, 3]) == 6
# assert exercise_3_subset_xor_sum([5, 1, 6]) == 28
# assert exercise_3_subset_xor_sum([3, 4, 5, 6, 7, 8]) == 480

## Bài 4 — Biến thể: tập con có tổng bằng `target`

Trả mọi tập con (phần tử phân biệt, số **dương**) có tổng đúng bằng `target`. Hãy **cắt tỉa**: sắp xếp trước, dừng khi tổng vượt `target`.

```python
subsets_with_sum([1, 2, 3, 4], 5) -> [[1, 4], [2, 3]]
```

In [26]:
def exercise_4_subsets_with_sum(nums: List[int], target: int) -> List[List[int]]:
    # TODO
    pass


# assert sorted(exercise_4_subsets_with_sum([1, 2, 3, 4], 5)) == [[1, 4], [2, 3]]
# assert sorted(exercise_4_subsets_with_sum([2, 3, 5], 5)) == [[2, 3], [5]]
# assert exercise_4_subsets_with_sum([7], 3) == []

## Bài 5 — Mini challenge: thứ tự Gray code

Sinh mọi tập con sao cho hai tập con **liên tiếp** chỉ khác nhau đúng **một** phần tử (thêm hoặc bớt một). Gợi ý nhanh: mã Gray `g = i ^ (i >> 1)` với `i = 0 .. 2^n − 1`; hai mã Gray liên tiếp khác nhau đúng một bit.

In [27]:
def exercise_5_subsets_gray(nums: List[int]) -> List[List[int]]:
    # TODO
    pass


# res = exercise_5_subsets_gray([1, 2, 3]); assert len(res) == 8 and len({tuple(sorted(s)) for s in res}) == 8
# res = exercise_5_subsets_gray([1, 2, 3]); assert all(len(set(a) ^ set(b)) == 1 for a, b in zip(res, res[1:]))

# 12. Gợi ý

**Bài 1**
- Mức 1: `4 = 100` → bit 2; `6 = 110` → bit 1 và 2.
- Mức 2: thứ tự cách lặp trùng với thứ tự mask (`[1,3]` ↔ mask `101`).

**Bài 3**
- Mức 1: duyệt mask, tính XOR, cộng dồn.
- Mức 2 (`O(n)`): mỗi bit có mặt trong phần tử nào đó sẽ bằng 1 ở **đúng một nửa** số tập con → đáp án = `(OR tất cả) << (n - 1)`.

**Bài 4**
- Mức 1: backtracking như 4.2, thêm tham số `remain`; `if nums[i] > remain: break` (đã sắp xếp).

**Bài 5**
- Mức 1: với `i` từ 0 tới `2^n − 1`: `g = i ^ (i >> 1)`, rồi đổi `g` thành tập con như bitmask.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [28]:
nums = [1, 2, 3]
for mask in (4, 6):
    print(f"mask {mask} = {mask:03b} ->", [nums[i] for i in range(3) if mask >> i & 1])
print("Vị trí của [1, 3] trong subsets_iter:", subsets_iter(nums).index([1, 3]))

mask 4 = 100 -> [3]
mask 6 = 110 -> [2, 3]
Vị trí của [1, 3] trong subsets_iter: 5


## Đáp án Bài 2

In [29]:
def answer_2_subsets_bitmask(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    result = []
    for mask in range(1 << n):
        result.append([nums[i] for i in range(n) if mask >> i & 1])
    return result

assert sorted(map(sorted, answer_2_subsets_bitmask([1, 2, 3]))) == [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
assert len(answer_2_subsets_bitmask(list(range(10)))) == 1024
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [30]:
from functools import reduce
from operator import xor, or_

def answer_3_subset_xor_sum(nums: List[int]) -> int:
    return sum(reduce(xor, s, 0) for s in subsets_bitmask(nums))

def subset_xor_sum_fast(nums: List[int]) -> int:
    return reduce(or_, nums, 0) << (len(nums) - 1)

assert answer_3_subset_xor_sum([1, 3]) == 6
assert answer_3_subset_xor_sum([5, 1, 6]) == 28
assert answer_3_subset_xor_sum([3, 4, 5, 6, 7, 8]) == 480
for _ in range(500):
    nums = [random.randint(1, 20) for _ in range(random.randint(1, 8))]
    assert answer_3_subset_xor_sum(nums) == subset_xor_sum_fast(nums)
print("Bài 3 PASS (và công thức O(n) khớp brute force)")

Bài 3 PASS (và công thức O(n) khớp brute force)

## Đáp án Bài 4

In [31]:
def answer_4_subsets_with_sum(nums: List[int], target: int) -> List[List[int]]:
    nums = sorted(nums)
    result, path = [], []
    def backtrack(start, remain):
        if remain == 0:
            result.append(path[:])
            return
        for i in range(start, len(nums)):
            if nums[i] > remain:
                break
            path.append(nums[i])
            backtrack(i + 1, remain - nums[i])
            path.pop()
    backtrack(0, target)
    return result

assert sorted(answer_4_subsets_with_sum([1, 2, 3, 4], 5)) == [[1, 4], [2, 3]]
assert sorted(answer_4_subsets_with_sum([2, 3, 5], 5)) == [[2, 3], [5]]
assert answer_4_subsets_with_sum([7], 3) == []
print("Bài 4 PASS")

Bài 4 PASS


Vì mọi số dương, khi `remain == 0` thì thêm số nào cũng vượt — nên ta `return` luôn.

## Đáp án Bài 5

In [32]:
def answer_5_subsets_gray(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    result = []
    for i in range(1 << n):
        g = i ^ (i >> 1)
        result.append([nums[j] for j in range(n) if g >> j & 1])
    return result

res = answer_5_subsets_gray([1, 2, 3])
print(res)
assert len(res) == 8 and len({tuple(sorted(s)) for s in res}) == 8
assert all(len(set(a) ^ set(b)) == 1 for a, b in zip(res, res[1:]))
print("Bài 5 PASS")

[[], [1], [1, 2], [2], [2, 3], [1, 2, 3], [1, 3], [3]]
Bài 5 PASS


# 14. Tổng kết

> `2^n` tập con. Ba cách: **lặp** (mỗi phần tử nhân đôi danh sách), **backtracking** (lưu ở **mọi** nút của cây tổ hợp), **bitmask** (`mask` từ `0` tới `2^n − 1`).

---

## Checklist trước khi nộp

- [ ] Có tập rỗng `[]` trong kết quả?
- [ ] Lưu bản sao `path[:]`, tạo list mới `s + [x]`?
- [ ] Không sửa list trong lúc lặp qua nó?
- [ ] Kiểm tra bit bằng `mask >> i & 1` (không so `mask & (1 << i)` với 1)?